## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-14-02-03-00 +0000,wsj,California voters had a chance to impose the n...,https://www.wsj.com/politics/california-voters...
1,2026-11-14-02-01-00 +0000,wsj,FAA Lifts Nationwide Groundstop that Upended M...,https://www.wsj.com/us-news/faa-temporarily-ha...
2,2026-11-14-02-00-18 +0000,nyt,How the University of North Carolina Tried to ...,https://www.nytimes.com/2026/11/13/us/universi...
3,2026-11-14-02-00-00 +0000,wsj,Behind the Nvidia Affair Sits a Vast Chinese S...,https://www.wsj.com/world/nvidia-china-espiona...
4,2026-11-14-02-00-00 +0000,wsj,Powerful Trump Allies Aided a Chinese Vape Ope...,https://www.wsj.com/politics/policy/china-vape...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2406/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
31,nvidia,36
28,trump,27
454,huang,22
149,new,21
453,jensen,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
385,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...,126
252,2026-11-13-16-01-00 +0000,wsj,Treasury Secretary Scott Bessent said Howard L...,https://www.wsj.com/politics/policy/bessent-sa...,123
351,2026-11-13-10-00-00 +0000,wapo,How disgraced Nvidia CEO Jensen Huang became a...,https://www.washingtonpost.com/technology/inte...,121
388,2026-11-13-03-27-02 +0000,nyt,Trump Had Urged Nvidia’s Jensen Huang to Fight...,https://www.nytimes.com/2026/11/12/us/politics...,116
63,2026-11-13-23-12-22 +0000,nypost,Jensen Huang commemorative plaque gone from fa...,https://nypost.com/2026/11/13/business/jensen-...,110


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
385,126,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...
389,103,2026-11-13-03-27-00 +0000,wsj,"Sen. Chris Van Hollen (D., Md.) said he would ...",https://www.wsj.com/politics/policy/chris-van-...
252,101,2026-11-13-16-01-00 +0000,wsj,Treasury Secretary Scott Bessent said Howard L...,https://www.wsj.com/politics/policy/bessent-sa...
78,99,2026-11-13-22-44-32 +0000,nypost,FAA launched new AI ‘SMART’ tool to all air ro...,https://nypost.com/2026/11/13/us-news/faa-laun...
5,85,2026-11-14-02-00-00 +0000,wsj,The White House ordered officials this summer ...,https://www.wsj.com/politics/policy/china-vape...
159,61,2026-11-13-20-07-00 +0000,wsj,Rep. Lauren Boebert lost her deep-red Colorado...,https://www.wsj.com/politics/elections/lauren-...
332,54,2026-11-13-10-55-43 +0000,cbc,Microsoft reports progress in reviving some se...,https://www.cbc.ca/news/world/microsoft-outage...
135,45,2026-11-13-21-06-14 +0000,nypost,"Trump will have historic meeting with Xi, Puti...",https://nypost.com/2026/11/13/us-news/trump-wi...
49,43,2026-11-13-23-52-22 +0000,latimes,"LAX, other airports fill with stranded passeng...",https://www.latimes.com/california/story/2026-...
334,42,2026-11-13-10-43-38 +0000,nypost,Russia hit with second car bombing in 2 days a...,https://nypost.com/2026/11/13/world-news/russi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
